# Soccer Video Tracker and Analysis

This notebook provides a comprehensive solution for analyzing soccer videos, including:
- Object detection and tracking of players and the ball
- Camera movement estimation for position stabilization
- Perspective transformation for real-world distance calculations
- Player speed calculation
- Ball control estimation
- Visual overlays and telemetry

## 1. Setup: Install and Import Libraries

First, we install and import the necessary libraries for object detection, tracking, and video processing.

In [ ]:
# Install required libraries
# Uncomment the following lines if running for the first time
# !pip install ultralytics
# !pip install opencv-python
# !pip install supervision
# !pip install numpy

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
from collections import defaultdict
import time

## 2. Object Detection and Tracking

We use a pre-trained YOLO model (yolov8x.pt) to detect 'person' (class 0) and 'sports ball' (class 32) in the video frames. The ByteTrack tracker from the supervision library assigns unique IDs to players across frames.

In [ ]:
class ObjectDetector:
    """Handles object detection using YOLO model."""
    
    # COCO class IDs
    PERSON_CLASS_ID = 0
    SPORTS_BALL_CLASS_ID = 32
    
    def __init__(self, model_path: str = "yolov8x.pt"):
        """Initialize the YOLO model.
        
        Args:
            model_path: Path to the YOLO model weights.
        """
        self.model = YOLO(model_path)
        self.target_classes = [self.PERSON_CLASS_ID, self.SPORTS_BALL_CLASS_ID]
    
    def detect(self, frame: np.ndarray, conf_threshold: float = 0.3):
        """Detect objects in a frame.
        
        Args:
            frame: Input video frame.
            conf_threshold: Confidence threshold for detections.
            
        Returns:
            Tuple of (person_detections, ball_detections) as supervision Detections.
        """
        results = self.model(frame, conf=conf_threshold, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)
        
        # Filter for persons
        person_mask = detections.class_id == self.PERSON_CLASS_ID
        person_detections = detections[person_mask]
        
        # Filter for sports ball
        ball_mask = detections.class_id == self.SPORTS_BALL_CLASS_ID
        ball_detections = detections[ball_mask]
        
        return person_detections, ball_detections

In [ ]:
class PlayerTracker:
    """Tracks players across frames using ByteTrack."""
    
    def __init__(self):
        """Initialize the ByteTrack tracker."""
        self.tracker = sv.ByteTrack(
            track_activation_threshold=0.25,
            lost_track_buffer=30,
            minimum_matching_threshold=0.8,
            frame_rate=30
        )
        self.positions_history = defaultdict(list)
        self.max_history = 30  # Store last 30 positions for speed calculation
    
    def update(self, detections: sv.Detections) -> sv.Detections:
        """Update tracker with new detections.
        
        Args:
            detections: Detected persons.
            
        Returns:
            Detections with assigned tracker IDs.
        """
        tracked_detections = self.tracker.update_with_detections(detections)
        return tracked_detections
    
    def update_position_history(self, tracker_id: int, position: tuple, camera_offset: tuple = (0, 0)):
        """Store position history for speed calculation.
        
        Args:
            tracker_id: Unique player ID.
            position: (x, y) position in frame.
            camera_offset: (dx, dy) camera movement offset.
        """
        # Adjust position by camera movement
        adjusted_position = (
            position[0] - camera_offset[0],
            position[1] - camera_offset[1],
            time.time()
        )
        self.positions_history[tracker_id].append(adjusted_position)
        
        # Keep only recent history
        if len(self.positions_history[tracker_id]) > self.max_history:
            self.positions_history[tracker_id].pop(0)
    
    def get_position_history(self, tracker_id: int) -> list:
        """Get position history for a player."""
        return self.positions_history.get(tracker_id, [])

## 3. Camera Movement Estimator

We use Optical Flow (Lucas-Kanade method) to estimate camera movement between frames. This is essential for stabilizing player positions and calculating accurate speeds.

The process involves:
1. Detecting good features to track in the previous frame (`cv2.goodFeaturesToTrack`)
2. Computing optical flow to find where those features moved (`cv2.calcOpticalFlowPyrLK`)
3. Calculating the median displacement as camera movement

In [ ]:
class CameraMovementEstimator:
    """Estimates camera movement using optical flow."""
    
    def __init__(self):
        """Initialize optical flow parameters."""
        self.previous_gray = None
        
        # Parameters for goodFeaturesToTrack
        self.feature_params = {
            'maxCorners': 100,
            'qualityLevel': 0.3,
            'minDistance': 7,
            'blockSize': 7
        }
        
        # Parameters for Lucas-Kanade optical flow
        self.lk_params = {
            'winSize': (21, 21),
            'maxLevel': 3,
            'criteria': (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01)
        }
        
        self.cumulative_movement = np.array([0.0, 0.0])
    
    def estimate(self, frame: np.ndarray) -> tuple:
        """Estimate camera movement from the previous frame.
        
        Args:
            frame: Current video frame.
            
        Returns:
            Tuple of (movement_x, movement_y) in pixels.
        """
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        movement = np.array([0.0, 0.0])
        
        if self.previous_gray is not None:
            # Find good features to track in previous frame
            prev_points = cv2.goodFeaturesToTrack(
                self.previous_gray,
                mask=None,
                **self.feature_params
            )
            
            if prev_points is not None and len(prev_points) > 0:
                # Calculate optical flow
                next_points, status, _ = cv2.calcOpticalFlowPyrLK(
                    self.previous_gray,
                    gray,
                    prev_points,
                    None,
                    **self.lk_params
                )
                
                if next_points is not None:
                    # Select good points
                    good_old = prev_points[status.flatten() == 1]
                    good_new = next_points[status.flatten() == 1]
                    
                    if len(good_old) > 0:
                        # Calculate displacement
                        displacement = good_new - good_old
                        
                        # Use median to be robust against outliers
                        movement[0] = np.median(displacement[:, 0])
                        movement[1] = np.median(displacement[:, 1])
        
        self.previous_gray = gray.copy()
        self.cumulative_movement += movement
        
        return tuple(movement)
    
    def get_cumulative_movement(self) -> tuple:
        """Get cumulative camera movement since start."""
        return tuple(self.cumulative_movement)
    
    def reset(self):
        """Reset the estimator state."""
        self.previous_gray = None
        self.cumulative_movement = np.array([0.0, 0.0])

## 4. Perspective Transformation

To calculate real-world distances (in meters), we need to transform pixel coordinates to actual pitch coordinates. This requires defining keypoints on the soccer pitch that map screen pixels to real-world measurements.

A standard soccer pitch is approximately:
- Length: 100-110 meters
- Width: 64-75 meters

We use these reference dimensions to compute a pixels-to-meters conversion ratio.

In [ ]:
class PerspectiveTransformer:
    """Handles perspective transformation from pixels to real-world meters."""
    
    # Standard soccer pitch dimensions in meters
    PITCH_LENGTH = 105.0  # meters
    PITCH_WIDTH = 68.0    # meters
    
    def __init__(self):
        """Initialize the perspective transformer."""
        self.transform_matrix = None
        self.inverse_matrix = None
        self.pixels_per_meter = None
    
    def set_keypoints(self, source_points: np.ndarray, dest_points: np.ndarray):
        """Set keypoints for perspective transformation.
        
        Args:
            source_points: 4 points in screen coordinates (pixels).
            dest_points: 4 corresponding points in pitch coordinates (meters).
        """
        source_points = np.float32(source_points)
        dest_points = np.float32(dest_points)
        
        self.transform_matrix = cv2.getPerspectiveTransform(source_points, dest_points)
        self.inverse_matrix = cv2.getPerspectiveTransform(dest_points, source_points)
    
    def set_simple_scale(self, frame_width: int, frame_height: int, 
                         pitch_length: float = None, pitch_width: float = None):
        """Set a simple linear scale based on frame dimensions.
        
        This is a simplified approach when exact pitch keypoints are not available.
        
        Args:
            frame_width: Width of the video frame in pixels.
            frame_height: Height of the video frame in pixels.
            pitch_length: Visible pitch length in meters (default: full pitch).
            pitch_width: Visible pitch width in meters (default: full pitch).
        """
        if pitch_length is None:
            pitch_length = self.PITCH_LENGTH
        if pitch_width is None:
            pitch_width = self.PITCH_WIDTH
        
        # Calculate average pixels per meter
        pixels_per_meter_x = frame_width / pitch_length
        pixels_per_meter_y = frame_height / pitch_width
        
        self.pixels_per_meter = (pixels_per_meter_x + pixels_per_meter_y) / 2
    
    def transform_point(self, point: tuple) -> tuple:
        """Transform a point from pixels to meters.
        
        Args:
            point: (x, y) in pixels.
            
        Returns:
            (x, y) in meters.
        """
        if self.transform_matrix is not None:
            pt = np.array([[[point[0], point[1]]]], dtype=np.float32)
            transformed = cv2.perspectiveTransform(pt, self.transform_matrix)
            return (transformed[0][0][0], transformed[0][0][1])
        elif self.pixels_per_meter is not None:
            return (
                point[0] / self.pixels_per_meter,
                point[1] / self.pixels_per_meter
            )
        return point
    
    def pixel_distance_to_meters(self, pixel_distance: float) -> float:
        """Convert pixel distance to meters.
        
        Args:
            pixel_distance: Distance in pixels.
            
        Returns:
            Distance in meters.
        """
        if self.pixels_per_meter is not None:
            return pixel_distance / self.pixels_per_meter
        return pixel_distance / 10.0  # Default fallback ratio

## 5. Speed Calculation

Player speed is calculated based on:
1. Distance traveled between frames (in meters, using perspective transformation)
2. Time elapsed between measurements
3. Camera movement correction to account for panning/zooming

Speed is calculated in km/h for easier interpretation.

In [ ]:
class SpeedCalculator:
    """Calculates player speed from position history."""
    
    def __init__(self, perspective_transformer: PerspectiveTransformer):
        """Initialize speed calculator.
        
        Args:
            perspective_transformer: Transformer for pixel-to-meter conversion.
        """
        self.transformer = perspective_transformer
        self.smoothed_speeds = defaultdict(lambda: 0.0)
        self.smoothing_factor = 0.3  # Exponential smoothing
    
    def calculate_speed(self, position_history: list, window_frames: int = 5) -> float:
        """Calculate speed from position history.
        
        Args:
            position_history: List of (x, y, timestamp) tuples.
            window_frames: Number of frames to average over.
            
        Returns:
            Speed in km/h.
        """
        if len(position_history) < 2:
            return 0.0
        
        # Use recent positions
        recent = position_history[-min(window_frames, len(position_history)):]
        
        if len(recent) < 2:
            return 0.0
        
        # Calculate total distance and time
        total_distance_px = 0.0
        for i in range(1, len(recent)):
            dx = recent[i][0] - recent[i-1][0]
            dy = recent[i][1] - recent[i-1][1]
            total_distance_px += np.sqrt(dx**2 + dy**2)
        
        total_time = recent[-1][2] - recent[0][2]
        
        if total_time <= 0:
            return 0.0
        
        # Convert to meters
        total_distance_m = self.transformer.pixel_distance_to_meters(total_distance_px)
        
        # Calculate speed in m/s, then convert to km/h
        speed_ms = total_distance_m / total_time
        speed_kmh = speed_ms * 3.6
        
        # Clamp to reasonable values (max ~40 km/h for sprinting)
        speed_kmh = min(speed_kmh, 40.0)
        
        return speed_kmh
    
    def get_smoothed_speed(self, tracker_id: int, raw_speed: float) -> float:
        """Apply exponential smoothing to speed values.
        
        Args:
            tracker_id: Player tracker ID.
            raw_speed: Current raw speed value.
            
        Returns:
            Smoothed speed in km/h.
        """
        prev_speed = self.smoothed_speeds[tracker_id]
        smoothed = self.smoothing_factor * raw_speed + (1 - self.smoothing_factor) * prev_speed
        self.smoothed_speeds[tracker_id] = smoothed
        return smoothed

## 6. Ball Control Logic

We estimate which team has ball control based on:
1. Finding the closest player to the ball
2. Assigning that player's team as having control
3. Tracking control percentages over time

For team assignment, we use a simple heuristic based on player position (left/right half of field) or jersey color detection could be added for more accuracy.

In [ ]:
class BallControlEstimator:
    """Estimates team ball control based on ball-player proximity."""
    
    CONTROL_DISTANCE_THRESHOLD = 50  # pixels - player must be within this distance to have control
    
    def __init__(self):
        """Initialize ball control estimator."""
        self.team_control_frames = {1: 0, 2: 0}
        self.total_frames = 0
        self.current_controlling_player = None
        self.player_teams = {}  # Cache of player team assignments
    
    def assign_team(self, player_id: int, player_position: tuple, frame_width: int) -> int:
        """Assign a team to a player based on position.
        
        Simple heuristic: players on left half are Team 1, right half are Team 2.
        In production, jersey color detection would be more accurate.
        
        Args:
            player_id: Player tracker ID.
            player_position: (x, y) position.
            frame_width: Width of the frame.
            
        Returns:
            Team ID (1 or 2).
        """
        if player_id in self.player_teams:
            return self.player_teams[player_id]
        
        # Simple position-based assignment (could be replaced with color detection)
        team = 1 if player_position[0] < frame_width / 2 else 2
        self.player_teams[player_id] = team
        return team
    
    def update(
        self, 
        player_detections: sv.Detections, 
        ball_position: tuple,
        frame_width: int
    ) -> dict:
        """Update ball control state.
        
        Args:
            player_detections: Tracked player detections.
            ball_position: (x, y) position of the ball, or None if not detected.
            frame_width: Width of the frame.
            
        Returns:
            Dictionary with control info and percentages.
        """
        self.total_frames += 1
        controlling_team = None
        controlling_player = None
        min_distance = float('inf')
        
        if ball_position is not None and len(player_detections) > 0:
            # Find closest player to ball
            for i, bbox in enumerate(player_detections.xyxy):
                # Get player foot position (bottom center of bbox)
                player_x = (bbox[0] + bbox[2]) / 2
                player_y = bbox[3]  # Bottom of bounding box
                
                distance = np.sqrt(
                    (player_x - ball_position[0])**2 + 
                    (player_y - ball_position[1])**2
                )
                
                if distance < min_distance and distance < self.CONTROL_DISTANCE_THRESHOLD:
                    min_distance = distance
                    if player_detections.tracker_id is not None:
                        player_id = player_detections.tracker_id[i]
                        controlling_player = player_id
                        controlling_team = self.assign_team(
                            player_id, 
                            (player_x, player_y), 
                            frame_width
                        )
        
        # Update control counters
        if controlling_team is not None:
            self.team_control_frames[controlling_team] += 1
            self.current_controlling_player = controlling_player
        
        # Calculate percentages
        team1_pct = (self.team_control_frames[1] / max(1, self.total_frames)) * 100
        team2_pct = (self.team_control_frames[2] / max(1, self.total_frames)) * 100
        
        return {
            'controlling_team': controlling_team,
            'controlling_player': controlling_player,
            'team1_percentage': team1_pct,
            'team2_percentage': team2_pct
        }

## 7. Visualization

The visualization module handles drawing overlays on the video frames:
- Ellipses/circles under players
- Player IDs and speed (km/h)
- Camera movement X/Y display
- Team ball control percentages

In [ ]:
class Visualizer:
    """Handles all visualization overlays on video frames."""
    
    # Colors (BGR format)
    TEAM1_COLOR = (255, 100, 100)   # Light blue
    TEAM2_COLOR = (100, 100, 255)   # Light red
    BALL_COLOR = (0, 255, 255)      # Yellow
    TEXT_COLOR = (255, 255, 255)    # White
    OVERLAY_BG = (0, 0, 0)          # Black background
    
    def __init__(self):
        """Initialize visualizer with annotators."""
        self.ellipse_annotator = sv.EllipseAnnotator(
            color=sv.Color.from_hex("#00FF00"),
            thickness=2
        )
        self.label_annotator = sv.LabelAnnotator(
            text_position=sv.Position.TOP_CENTER,
            text_color=sv.Color.WHITE,
            color=sv.Color.from_hex("#000000")
        )
    
    def draw_player_ellipse(
        self, 
        frame: np.ndarray, 
        bbox: np.ndarray, 
        team: int = 1
    ) -> np.ndarray:
        """Draw ellipse under a player.
        
        Args:
            frame: Video frame.
            bbox: Bounding box [x1, y1, x2, y2].
            team: Team number for color selection.
            
        Returns:
            Annotated frame.
        """
        color = self.TEAM1_COLOR if team == 1 else self.TEAM2_COLOR
        
        # Calculate ellipse parameters
        center_x = int((bbox[0] + bbox[2]) / 2)
        center_y = int(bbox[3])  # Bottom of bounding box
        width = int((bbox[2] - bbox[0]) / 2)
        height = int(width / 3)  # Flattened ellipse
        
        cv2.ellipse(
            frame,
            (center_x, center_y),
            (width, height),
            0, 0, 360,
            color,
            2
        )
        
        return frame
    
    def draw_player_info(
        self, 
        frame: np.ndarray, 
        bbox: np.ndarray, 
        player_id: int, 
        speed_kmh: float
    ) -> np.ndarray:
        """Draw player ID and speed.
        
        Args:
            frame: Video frame.
            bbox: Bounding box.
            player_id: Player tracker ID.
            speed_kmh: Player speed in km/h.
            
        Returns:
            Annotated frame.
        """
        center_x = int((bbox[0] + bbox[2]) / 2)
        top_y = int(bbox[1]) - 10
        
        # Draw background rectangle
        text = f"ID:{player_id} {speed_kmh:.1f}km/h"
        (text_width, text_height), _ = cv2.getTextSize(
            text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1
        )
        
        cv2.rectangle(
            frame,
            (center_x - text_width // 2 - 5, top_y - text_height - 5),
            (center_x + text_width // 2 + 5, top_y + 5),
            self.OVERLAY_BG,
            -1
        )
        
        cv2.putText(
            frame,
            text,
            (center_x - text_width // 2, top_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            self.TEXT_COLOR,
            1
        )
        
        return frame
    
    def draw_ball(self, frame: np.ndarray, position: tuple) -> np.ndarray:
        """Draw ball marker.
        
        Args:
            frame: Video frame.
            position: (x, y) ball position.
            
        Returns:
            Annotated frame.
        """
        cv2.circle(
            frame,
            (int(position[0]), int(position[1])),
            10,
            self.BALL_COLOR,
            2
        )
        return frame
    
    def draw_camera_movement_overlay(
        self, 
        frame: np.ndarray, 
        movement_x: float, 
        movement_y: float
    ) -> np.ndarray:
        """Draw camera movement overlay.
        
        Args:
            frame: Video frame.
            movement_x: Camera X movement in pixels.
            movement_y: Camera Y movement in pixels.
            
        Returns:
            Annotated frame.
        """
        overlay_text = f"Camera Movement X: {movement_x:.1f} Y: {movement_y:.1f}"
        
        # Draw background
        cv2.rectangle(frame, (10, 10), (350, 40), self.OVERLAY_BG, -1)
        
        cv2.putText(
            frame,
            overlay_text,
            (20, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            self.TEXT_COLOR,
            2
        )
        
        return frame
    
    def draw_ball_control_overlay(
        self, 
        frame: np.ndarray, 
        team1_pct: float, 
        team2_pct: float
    ) -> np.ndarray:
        """Draw team ball control percentages overlay.
        
        Args:
            frame: Video frame.
            team1_pct: Team 1 ball control percentage.
            team2_pct: Team 2 ball control percentage.
            
        Returns:
            Annotated frame.
        """
        h, w = frame.shape[:2]
        
        # Draw background
        cv2.rectangle(frame, (10, 50), (250, 110), self.OVERLAY_BG, -1)
        
        # Title
        cv2.putText(
            frame,
            "Team Ball Control",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            self.TEXT_COLOR,
            1
        )
        
        # Team 1
        cv2.putText(
            frame,
            f"Team 1: {team1_pct:.1f}%",
            (20, 90),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            self.TEAM1_COLOR,
            1
        )
        
        # Team 2
        cv2.putText(
            frame,
            f"Team 2: {team2_pct:.1f}%",
            (130, 90),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            self.TEAM2_COLOR,
            1
        )
        
        return frame
    
    def annotate_frame(
        self,
        frame: np.ndarray,
        player_detections: sv.Detections,
        ball_position: tuple,
        player_speeds: dict,
        player_teams: dict,
        camera_movement: tuple,
        ball_control: dict
    ) -> np.ndarray:
        """Apply all annotations to a frame.
        
        Args:
            frame: Video frame.
            player_detections: Tracked player detections.
            ball_position: Ball (x, y) position or None.
            player_speeds: Dict of player_id -> speed.
            player_teams: Dict of player_id -> team.
            camera_movement: (x, y) camera movement.
            ball_control: Ball control info dict.
            
        Returns:
            Fully annotated frame.
        """
        annotated = frame.copy()
        frame_width = frame.shape[1]
        
        # Draw players
        if len(player_detections) > 0:
            for i, bbox in enumerate(player_detections.xyxy):
                if player_detections.tracker_id is not None:
                    player_id = player_detections.tracker_id[i]
                    team = player_teams.get(player_id, 1)
                    speed = player_speeds.get(player_id, 0.0)
                    
                    annotated = self.draw_player_ellipse(annotated, bbox, team)
                    annotated = self.draw_player_info(annotated, bbox, player_id, speed)
        
        # Draw ball
        if ball_position is not None:
            annotated = self.draw_ball(annotated, ball_position)
        
        # Draw overlays
        annotated = self.draw_camera_movement_overlay(
            annotated, 
            camera_movement[0], 
            camera_movement[1]
        )
        
        annotated = self.draw_ball_control_overlay(
            annotated,
            ball_control['team1_percentage'],
            ball_control['team2_percentage']
        )
        
        return annotated

## 8. Main Soccer Video Processor

The main processor class integrates all components to process soccer videos end-to-end.

In [ ]:
class SoccerVideoProcessor:
    """Main class for processing soccer videos."""
    
    def __init__(self, model_path: str = "yolov8x.pt"):
        """Initialize all processing components.
        
        Args:
            model_path: Path to YOLO model weights.
        """
        self.detector = ObjectDetector(model_path)
        self.tracker = PlayerTracker()
        self.camera_estimator = CameraMovementEstimator()
        self.perspective_transformer = PerspectiveTransformer()
        self.speed_calculator = SpeedCalculator(self.perspective_transformer)
        self.ball_control = BallControlEstimator()
        self.visualizer = Visualizer()
    
    def process_frame(self, frame: np.ndarray) -> np.ndarray:
        """Process a single video frame.
        
        Args:
            frame: Input video frame.
            
        Returns:
            Annotated video frame.
        """
        frame_height, frame_width = frame.shape[:2]
        
        # Estimate camera movement
        camera_movement = self.camera_estimator.estimate(frame)
        
        # Detect objects
        person_detections, ball_detections = self.detector.detect(frame)
        
        # Track players
        tracked_players = self.tracker.update(person_detections)
        
        # Get ball position
        ball_position = None
        if len(ball_detections) > 0:
            bbox = ball_detections.xyxy[0]
            ball_position = ((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)
        
        # Update position history and calculate speeds
        player_speeds = {}
        cumulative_camera = self.camera_estimator.get_cumulative_movement()
        
        if tracked_players.tracker_id is not None:
            for i, tracker_id in enumerate(tracked_players.tracker_id):
                bbox = tracked_players.xyxy[i]
                position = ((bbox[0] + bbox[2]) / 2, bbox[3])  # Foot position
                
                # Update position history with camera correction
                self.tracker.update_position_history(
                    tracker_id, 
                    position, 
                    cumulative_camera
                )
                
                # Calculate speed
                history = self.tracker.get_position_history(tracker_id)
                raw_speed = self.speed_calculator.calculate_speed(history)
                smoothed_speed = self.speed_calculator.get_smoothed_speed(tracker_id, raw_speed)
                player_speeds[tracker_id] = smoothed_speed
        
        # Update ball control
        control_info = self.ball_control.update(
            tracked_players, 
            ball_position, 
            frame_width
        )
        
        # Create annotated frame
        annotated_frame = self.visualizer.annotate_frame(
            frame,
            tracked_players,
            ball_position,
            player_speeds,
            self.ball_control.player_teams,
            camera_movement,
            control_info
        )
        
        return annotated_frame
    
    def process_video(
        self, 
        input_path: str, 
        output_path: str = None,
        display: bool = False,
        max_frames: int = None
    ):
        """Process a video file.
        
        Args:
            input_path: Path to input video file.
            output_path: Path to save output video (optional).
            display: Whether to display video during processing.
            max_frames: Maximum number of frames to process (None for all).
        """
        cap = cv2.VideoCapture(input_path)
        
        if not cap.isOpened():
            raise ValueError(f"Could not open video file: {input_path}")
        
        # Get video properties
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"Video: {width}x{height} @ {fps}fps, {total_frames} frames")
        
        # Set up perspective transformer with simple scale
        self.perspective_transformer.set_simple_scale(width, height)
        
        # Set up video writer if output path provided
        writer = None
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        
        frame_count = 0
        
        try:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                
                if max_frames and frame_count >= max_frames:
                    break
                
                # Process frame
                annotated_frame = self.process_frame(frame)
                
                # Write output
                if writer:
                    writer.write(annotated_frame)
                
                # Display if requested
                if display:
                    cv2.imshow('Soccer Tracker', annotated_frame)
                    if cv2.waitKey(1) & 0xFF == ord('q'):
                        break
                
                frame_count += 1
                
                if frame_count % 100 == 0:
                    print(f"Processed {frame_count}/{total_frames} frames")
        
        finally:
            cap.release()
            if writer:
                writer.release()
            if display:
                cv2.destroyAllWindows()
        
        print(f"Processing complete. Processed {frame_count} frames.")

## 9. Main Execution Block

Define the video path and run the processor to analyze the soccer video.

In [ ]:
if __name__ == "__main__":
    # Configuration
    VIDEO_PATH = "soccer_match.mp4"  # Path to input video
    OUTPUT_PATH = "soccer_output.mp4"  # Path to output video
    MODEL_PATH = "yolov8x.pt"  # YOLO model path
    
    # Create processor
    processor = SoccerVideoProcessor(model_path=MODEL_PATH)
    
    # Process video
    # Set display=True to see video during processing (requires display)
    # Set max_frames to limit processing for testing
    processor.process_video(
        input_path=VIDEO_PATH,
        output_path=OUTPUT_PATH,
        display=False,
        max_frames=None  # Process all frames
    )
    
    print(f"Output saved to: {OUTPUT_PATH}")

## 10. Usage Notes

### Prerequisites
1. Install required libraries:
   ```bash
   pip install ultralytics opencv-python supervision numpy
   ```

2. Download a YOLO model (automatically downloaded on first use):
   - yolov8n.pt (smallest, fastest)
   - yolov8s.pt (small)
   - yolov8m.pt (medium)
   - yolov8l.pt (large)
   - yolov8x.pt (extra large, most accurate)

### Running the Notebook
1. Update `VIDEO_PATH` to point to your soccer video file.
2. Run all cells in order.
3. The output video will be saved to `OUTPUT_PATH`.

### Customization
- **Perspective Transformation**: For more accurate distance calculations, define actual pitch keypoints using `perspective_transformer.set_keypoints()`.
- **Team Assignment**: Replace the simple position-based team assignment with jersey color detection for more accurate team identification.
- **Speed Calculation**: Adjust the `window_frames` parameter in `SpeedCalculator.calculate_speed()` for smoother or more responsive speed values.
- **Ball Control Threshold**: Modify `BallControlEstimator.CONTROL_DISTANCE_THRESHOLD` to adjust when a player is considered to have ball control.